# 知识架构系列 2：上下文的崩塌 —— 时序混乱与记忆漂移
### *Alex 的新困扰：当“事实”开始打架*

Alex 发现，虽然 Agnes/GPT-4o 这一代模型已经很难在物理位置上“丢失”信息（Lost in the middle 已得到显著优化），但它们在处理**时序事实冲突 (Temporal Contradiction)** 时依然非常脆弱。

**本章实验设计：**
1. **模拟极长对话**：构造 **200 轮**（共 400 条）业务对话，模拟一个月的审计周期。
2. **引入时序阶梯 (Temporal Ladder)**：
    - **Day 1**: 认定项目为“跨境”，罚金 35%。
    - **Day 10 (中段)**: 修正为“境内”，罚金 0%。
    - **Day 20 (近端)**: 再次修正为“特殊特区业务”，罚金 10%。
3. **压力测试**：在 400 条消息的背景下，问 AI “当前最新的罚金标准是多少？”
4. **核心痛点**：在全量堆叠的上下文中，AI 必须在 3 个相互矛盾的事实中通过推理找到“最后生效”的那一个。这种“逻辑检索”的成本和错误率远高于静态知识查询。

In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv(override=True)
model = init_chat_model(model="agnes-2.0-flash", model_provider="openai")

print("AI 助手已就绪，准备发起 200 轮时序压力测试。")

## 1. 构造 200 轮“时序陷阱”对话
我们将对话分为前、中、后三个阶段，每个阶段都注入了互相矛盾的业务决策。

In [ ]:
history = []

# --- 阶段 1：初始决策 (Round 1) ---
history.append(HumanMessage(content="【Day 1 决策】：项目 CYBER-TIGER 确认为‘跨境业务’，适用 35% 罚金规则。"))
history.append(AIMessage(content="收到，当前标准：跨境，罚金 35%。"))

# 填充 50 轮行政杂事
for i in range(2, 51):
    history.append(HumanMessage(content=f"轮次 {i}：关于办公室绿植的灌溉频率需要调整。"))
    history.append(AIMessage(content=f"已记录轮次 {i} 的杂务：绿植维护说明已更新。"))

# --- 阶段 2：第一次时序修正 (Round 51 - 位于中间) ---
history.append(HumanMessage(content="【Day 10 重要修正】：法务部撤销了之前的认定。CYBER-TIGER 现在改为‘全境内业务’，罚金规则变更为 0%。之前的 35% 记录废弃。"))
history.append(AIMessage(content="收到 Day 10 修正：该项目现在是‘全境内’，罚金已清零。"))

# 再填充 100 轮行政噪声（模拟真实业务中极长的讨论链路）
for i in range(52, 151):
    history.append(HumanMessage(content=f"轮次 {i}：讨论关于 GC-{2000+i} 号服务器的散热风扇转速问题。"))
    history.append(AIMessage(content=f"已记录轮次 {i} 的散热数据。"))

# --- 阶段 3：第二次时序修正 (Round 151 - 靠近末尾) ---
history.append(HumanMessage(content="【Day 20 最终终审】：根据最新的《特区法》，CYBER-TIGER 被重新定义为‘特区准跨境业务’。罚金标准固定为 10%。这是目前的最终版本。"))
history.append(AIMessage(content="确认 Day 20 终审：当前最终身份为‘特区业务’，罚金标准 10%。"))

# 最后 50 轮收尾讨论
for i in range(152, 201):
    history.append(HumanMessage(content=f"轮次 {i}：确认明天会议的订餐单号是 Order-{3000+i}。"))
    history.append(AIMessage(content=f"已记录轮次 {i} 的订餐信息。"))

print(f"时序对话构造完成。总消息数: {len(history)} 条。")
print("当前状态：上下文中存在三个版本的罚金标准 (35%, 0%, 10%)，且位置分散在开头、中间和结尾。")

## 2. 压力测试：AI 能在冲突中定位“现在”吗？
我们不仅问 AI 结果，还要求它解释为什么选择这个结果。如果它受“第一印象（Primacy Bias）”影响，会说 35%；如果受“干扰（Interference）”影响，可能会混淆标准。

In [ ]:
query = "请告诉我：根据我们这 200 轮对话的所有记录，CYBER-TIGER 项目目前‘当前最准确’的罚金比例是多少？并请说明你是如何从历史记录中推导出这个‘最新’结论的。"

print("--- 正在进行 200 轮长对话时序推理 ---\n")
response = model.invoke(history + [HumanMessage(content=query)])

print("AI 的最终研判：\n")
print(response.content)

## 3. 深度复盘：为什么这会导致企业风险？

### 现象分析：逻辑熵增与时序污染
1. **逻辑熵增**：即使模型能“看到”所有信息，但随着对话增长，上下文中存在的**陈旧事实（Stale Facts）**比例越来越高。AI 必须在每一轮对话中耗费巨大的算力去甄别“哪一句话是废弃的”。
2. **推理漂移**：当规则 1、规则 2、规则 3 同时出现在 1M 的窗口内，模型的自注意力机制（Self-Attention）会被这些相似的关键词（CYBER-TIGER, 罚金）牵引。在极高负载下，AI 可能会“缝合”这些事实，例如得出“特区境内业务，罚金 35%”这种错误的组合。
3. **时序盲区**：在大模型眼中，上下文是**平面的**，它没有天然的“时间流逝”概念，除非它非常刻意地去对比时间戳。这与人类大脑不同——我们会将旧的知识“覆盖”或“归档”。

### Alex 的终极领悟：我们需要“事实归档”
Alex 意识到，与其让 AI 在 400 条消息中反复纠结哪个是最新版本，不如：
- 每次发生变更时，由 Graphiti 捕捉并更新图谱。
- Graphiti 具有 **Bi-temporal (双时序)** 特性：当新事实（10% 罚金）存入时，它会自动将旧事实（0% 和 35%）标记为 `invalidated`（已失效）。
- **归档后的状态**：上下文窗口被清空，只剩下一张干净的、永远代表“当前真实状态”的图谱。

--- 
### 下一章预告
在 **Notebook 3** 中，我们将演示如何将这 200 轮混乱的对话通过 **Graphiti** 进行归档。你会看到，几百条垃圾信息是如何被压缩成一个“始终正确”的动态事实节点的。